# EN4554 – Assignment 2 on CNNs

**Name:** SULOJAN R.  
**Index No.:** 220626V
**GitHub:** TODO (link to profile / repository)

---
**Claude / Copilot acknowledgement:** TODO (state which tools were used and for what; you must fully understand all code).

**Contents**
- [Setup](#setup)
- [Task 1: Backpropagation](#task-1-backpropagation)
- [Task 2: Optimizers](#task-2-optimizers)
- [Task 3: Regularization and residual connections](#task-3-regularization-and-residual-connections)
- [Task 4: CNN (Optional)](#task-4-cnn-optional)
- [Conclusion](#conclusion)

In [1]:
import time
import copy
import random
import os

from google.colab import drive
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import TensorDataset, DataLoader, random_split

In [2]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [3]:
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
set_seed(SEED)

In [4]:
drive.mount('/content/drive', force_remount=True)
DATA = '/content/drive/MyDrive/datasets/cifar10'
os.makedirs(DATA, exist_ok=True)
torchvision.datasets.CIFAR10(root=DATA, train=True, download=True)

Mounted at /content/drive


Dataset CIFAR10
    Number of datapoints: 50000
    Root location: /content/drive/MyDrive/datasets/cifar10
    Split: Train

### Data loading, split and normalization

In [5]:
def get_cifar10_splits(root=DATA, n_train=45_000, n_val=5_000, seed=SEED):
    full_train = torchvision.datasets.CIFAR10(root=root, train=True, download=False)
    test_set = torchvision.datasets.CIFAR10(root=root, train=False, download=False)
    g = torch.Generator().manual_seed(seed)
    train_set, val_set = random_split(full_train, [n_train, n_val], generator=g)
    return train_set, val_set, test_set

def compute_channel_stats(train_subset):
    data = train_subset.dataset.data[train_subset.indices] / 255.0
    return data.mean(axis=(0, 1, 2)), data.std(axis=(0, 1, 2))

train_data, val_data, test_data = get_cifar10_splits()
mean, std = compute_channel_stats(train_data)
print(f"Calculated per channel training statistics: Mean - {mean}, STD - {std}")

Calculated per channel training statistics: Mean - [0.49165571 0.48233405 0.44667923], STD - [0.24708116 0.24350869 0.26158066]


In [6]:
MEAN = torch.tensor(mean, dtype=torch.float32).view(1, 3, 1, 1)
STD  = torch.tensor(std,  dtype=torch.float32).view(1, 3, 1, 1)

def to_normalized_tensor(x_uint8):
    x = torch.from_numpy(x_uint8).float().div(255).permute(0, 3, 1, 2)
    return (x - MEAN) / STD

full = train_data.dataset
X_all, y_all = full.data, np.array(full.targets)

X_train = to_normalized_tensor(X_all[train_data.indices])
y_train = torch.from_numpy(y_all[train_data.indices])
X_val   = to_normalized_tensor(X_all[val_data.indices])
y_val   = torch.from_numpy(y_all[val_data.indices])
X_test  = to_normalized_tensor(test_data.data)
y_test  = torch.tensor(test_data.targets)

BATCH_SIZE = 128
train_loader = DataLoader(
    TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True,  pin_memory=True
)
val_loader = DataLoader(
    TensorDataset(X_val, y_val), batch_size=512, shuffle=False, pin_memory=True
)
test_loader = DataLoader(
    TensorDataset(X_test,  y_test), batch_size=512, shuffle=False, pin_memory=True
)
CLASSES = full.classes

### Common utilities (training / evaluation loops)

In [7]:
def evaluate(model, loader, criterion=None, device=DEVICE):
    """Return (mean_loss, accuracy) over a loader."""
    criterion = criterion or nn.CrossEntropyLoss()
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            logits = model(x)
            total_loss += criterion(logits, y).item() * y.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            n += y.size(0)
    return total_loss / n, correct / n

def train_one_epoch(model, loader, optimizer, criterion, augment_fn=None, batch_scheduler=None, device=DEVICE):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        y_true = y if y.ndim == 1 else y.argmax(1)
        if augment_fn is not None:
            x, y = augment_fn(x, y)             

        logits = model(x)
        loss = criterion(logits, y)                
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        if batch_scheduler is not None:
            batch_scheduler.step()

        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y_true).sum().item()
        n += x.size(0)
    return total_loss / n, correct / n

def fit(model, train_loader, val_loader, optimizer, epochs, criterion=None,
        scheduler=None, batch_scheduler=None, augment_fn=None, verbose=True, device=DEVICE):
    """Train for `epochs`; return history dict of per-epoch lists."""
    criterion = criterion or nn.CrossEntropyLoss()
    model.to(device)
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "time": []}

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion,
                                          augment_fn, batch_scheduler)
        va_loss, va_acc = evaluate(model, val_loader, criterion)
        if scheduler is not None:         
            scheduler.step()
        dt = time.time() - t0

        for k, v in zip(history, (tr_loss, tr_acc, va_loss, va_acc, dt)):
            history[k].append(v)
        if verbose:
            print(f"ep {epoch:3d} | train {tr_loss:.4f} / {tr_acc*100:5.2f}% "
                  f"| val {va_loss:.4f} / {va_acc*100:5.2f}% | {dt:.1f}s")
    return history


def count_parameters(model):
    """Number of trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def plot_history(histories: dict, keys=("train_loss", "val_acc"), title=None):
    """histories: {run_name: history_dict}. One subplot per key, one line per run."""
    fig, axes = plt.subplots(1, len(keys), figsize=(6 * len(keys), 4), squeeze=False)
    for ax, key in zip(axes[0], keys):
        for name, h in histories.items():
            y = np.array(h[key]) * (100 if "acc" in key else 1)
            ax.plot(range(1, len(y) + 1), y, label=name)
        ax.set_xlabel("epoch")
        ax.set_ylabel(key + (" (%)" if "acc" in key else ""))
        ax.set_title(key)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    if title:
        fig.suptitle(title)
    fig.tight_layout()
    plt.show()
    return fig

---
## Task 1: Backpropagation

### 1(a) Linear, ReLU and softmax cross-entropy with explicit forward / backward (no autograd)

Implement each layer as a class with `forward` (caching what is needed) and `backward` (returns gradient w.r.t. input and stores parameter gradients). Then assemble an MLP.

**Derivations (write briefly, LaTeX):**
- Linear: $y = xW + b$ → $\partial L/\partial W = \ldots$, $\partial L/\partial b = \ldots$, $\partial L/\partial x = \ldots$
- ReLU: TODO
- Softmax cross-entropy (combined): TODO

In [8]:
class Linear:
    def __init__(self, in_features, out_features, device=DEVICE, dtype=torch.float32):
        self.W = torch.randn(in_features, out_features, device=device, dtype=dtype) * np.sqrt(2 / in_features)
        self.b = torch.zeros(out_features, device=device, dtype=dtype)
        self.dW = torch.zeros_like(self.W)
        self.db = torch.zeros_like(self.b)
    def forward(self, x):
        self.x = x
        return x @ self.W + self.b
    def backward(self, grad_out):
        self.dW = self.x.T @ grad_out
        self.db = grad_out.sum(dim=0)
        return grad_out @ self.W.T
        
    def params(self):
        return [(self.W, self.dW), (self.b, self.db)]

class ReLU:
    def forward(self, x):
        self.mask = x > 0
        return x * self.mask
    def backward(self, grad_out):
        return grad_out * self.mask
    def params(self):
        return []
    
class Sigmoid:
    def forward(self, x):
        self.s = torch.sigmoid(x)
        return self.s
    def backward(self, grad_out):
        return grad_out * self.s * (1 - self.s)
    def params(self):
        return []
    
class SoftmaxCrossEntropy:
    def forward(self, logits, targets):
        N = logits.shape[0]
        z = logits - logits.max(dim=1, keepdim=True).values
        log_p = z - torch.logsumexp(z, dim=1, keepdim=True)
        self.p, self.y, self.N = log_p.exp(), targets, N
        return -log_p[torch.arange(N, device=logits.device), targets].mean()
    def backward(self):
        dlogits = self.p.clone()
        dlogits[torch.arange(self.N, device=dlogits.device), self.y] -= 1.0
        return dlogits / self.N

In [9]:
class MLP:
    def __init__(self, layer_sizes, activation=ReLU, device=DEVICE, dtype=torch.float32):
        self.layers = []
        for in_, out_ in zip(layer_sizes[:-1], layer_sizes[1:]):
            self.layers.append(Linear(in_, out_, device=device, dtype=dtype))
            self.layers.append(activation())
        self.layers.pop()
    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x
    def backward(self, grad):
        for rev_layer in reversed(self.layers):
            grad = rev_layer.backward(grad)
        return grad
    def params(self):
        return [pv for layer in self.layers for pv in layer.params()]
    def linear_layers(self):
        return [l for l in self.layers if isinstance(l, Linear)]

### 1(b) Gradient verification: autograd and centred finite differences

In [10]:
def autograd_reference(mlp, x, y):
    """Gradients via autograd"""
    leaves = []
    for k, l in enumerate(mlp.linear_layers()):
        W = l.W.detach().clone().requires_grad_()
        b = l.b.detach().clone().requires_grad_()
        leaves += [W, b]
        x = x @ W + b
        if k < len(mlp.linear_layers()) - 1:
            x = torch.relu(x)
    loss = F.cross_entropy(x, y)
    loss.backward()
    return [t.grad for t in leaves]

def finite_difference_grad(loss_fn, param, eps=1e-5, n_checks=50):
    flat = param.view(-1)
    idx = torch.randperm(flat.numel())[:n_checks]
    grads = torch.zeros(len(idx), dtype=flat.dtype)
    for k, i in enumerate(idx):
        w = flat[i].item()
        flat[i] = w + eps
        L_pos = loss_fn()
        flat[i] = w - eps
        L_neg = loss_fn()
        flat[i] = w
        grads[k] = (L_pos - L_neg) / (2 * eps)
    return grads, idx

def relative_error(a, b):
    return (a - b).norm() / max(a.norm() + b.norm(), 1e-12)

In [11]:
set_seed()
mlp64 = MLP([3072, 16, 10], dtype=torch.float64)
ce64 = SoftmaxCrossEntropy()
xb, yb = next(iter(train_loader))
x64 = xb[:8].view(8, -1).to(DEVICE, torch.float64)
y64 = yb[:8].to(DEVICE)

# manual gradients
ce64.forward(mlp64.forward(x64), y64)
mlp64.backward(ce64.backward())
manual = [g.clone() for _, g in mlp64.params()]

# autograd gradients
auto = autograd_reference(mlp64, x64, y64)

# finite differences (subset of entries per parameter)
loss_fn = lambda: ce64.forward(mlp64.forward(x64), y64).item()
names = ["W1", "b1", "W2", "b2"]
rows = []
for name, (p, _), g_man, g_auto in zip(names, mlp64.params(), manual, auto):
    g_fd, idx = finite_difference_grad(loss_fn, p, eps=1e-5, n_checks=50)
    g_man_sub = g_man.view(-1)[idx].cpu()
    rows.append({
        "param": name,
        "shape": tuple(p.shape),
        "rel_err (manual vs autograd)": relative_error(g_man, g_auto).item(),
        "allclose autograd": torch.allclose(g_man, g_auto, rtol=1e-6, atol=1e-8),
        "rel_err (manual vs FD)": relative_error(g_man_sub, g_fd).item(),
        "allclose FD": torch.allclose(g_man_sub, g_fd, rtol=1e-4, atol=1e-6),
    })

pd.set_option("display.float_format", "{:.2e}".format)
pd.DataFrame(rows)

,param,shape,rel_err (manual vs autograd),allclose autograd,rel_err (manual vs FD),allclose FD
0,W1,"(3072, 16)",0.00e+00,True,8.22e-11,True
1,b1,"(16,)",0.00e+00,True,7.48e-11,True
2,W2,"(16, 10)",0.00e+00,True,9.84e-11,True
3,b2,"(10,)",0.00e+00,True,4.98e-11,True


#### 📝 Interpretation of gradient check

> TODO: write your interpretation here (a few sentences; this carries marks).
>
- *How small are the relative errors? Which is more accurate, autograd or finite differences, and why?*
- *Why float64? Any ReLU kinks that cause finite-difference mismatches?*

### 1(c) Train the MLP with SGD on flattened CIFAR-10

In [ ]:
# TODO: flatten images to 3072-d, choose architecture / lr / batch size / epochs
# TODO: manual SGD update loop using mlp.params() (no autograd, no torch.optim)
# TODO: record per-iteration / per-epoch loss, validation accuracy

In [ ]:
# TODO: plot training loss (and val loss/accuracy); report final validation accuracy

#### 📝 Interpretation of MLP training

> TODO: write your interpretation here (a few sentences; this carries marks).
>
- *Convergence behaviour, val accuracy vs. what a CNN would get, overfitting?*
- *Effect of learning rate / batch size you chose.*

### 1(d) Per-layer gradient norm at initialization in a 10-layer MLP: sigmoid vs. ReLU

In [ ]:
# TODO: build 10-layer MLPs with sigmoid and with ReLU (you may need a Sigmoid layer, explicit forward/backward)
# TODO: at initialization, one batch: compute ||dL/dW_l|| for each layer l
# TODO: plot per-layer gradient norm (log y-axis) for both activations in one figure

#### 📝 Explanation: vanishing / exploding gradients

> TODO: write your interpretation here (a few sentences; this carries marks).
>
- *Why do sigmoid gradients shrink with depth (σ'(x) ≤ 0.25)?*
- *Why does ReLU behave differently? What role does the initialization scale play (Xavier vs. Kaiming)?*

---
## Task 2: Optimizers

### 2(a) SGD with momentum and Adam as `torch.optim.Optimizer` subclasses

**Update rules (write in LaTeX):**
- SGD with momentum: TODO (match PyTorch's convention, state it)
- Adam with bias correction: TODO

In [ ]:
class MySGD(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-2, momentum=0.0):
        # TODO
        pass
    @torch.no_grad()
    def step(self, closure=None):
        pass

class MyAdam(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        # TODO
        pass
    @torch.no_grad()
    def step(self, closure=None):
        pass

In [ ]:
# TODO: show equivalence with torch.optim.SGD (momentum) and torch.optim.Adam
#  - same init weights, same data, several steps
#  - compare parameters after N steps (max abs diff + torch.allclose)

#### 📝 Interpretation of optimizer equivalence

> TODO: write your interpretation here (a few sentences; this carries marks).
>
- *Max difference observed; any conventions (dampening, eps placement) that could cause differences?*

### 2(b) Optimizer trajectories on $f(x,y)=\tfrac12(x^2+50y^2)$

In [ ]:
def f(xy):
    pass

# TODO: run SGD, SGD+momentum, Adam from the same start point for T steps; record trajectories
#       (state start point, learning rates, momentum, number of steps)

In [ ]:
# TODO: contour plot of f with the three trajectories overlaid (plt.contour); legend, axis labels

#### 📝 Explanation of trajectory differences

> TODO: write your interpretation here (a few sentences; this carries marks).
>
- *Ill-conditioning (condition number 50): oscillation along y for SGD.*
- *Effect of momentum; why Adam's per-coordinate scaling behaves differently.*
- *Limits of this toy example (learning rate sensitivity).*

### 2(c) Small CNN with each optimizer at three learning rates

In [ ]:
class SmallCNN(nn.Module):
    """TODO: small CNN for CIFAR-10 (state architecture + parameter count)."""
    def __init__(self):
        super().__init__()
        pass
    def forward(self, x):
        pass

In [ ]:
# TODO: grid: {SGD, SGD+momentum, Adam} x 3 learning rates each (state the values & epochs, same seed/init)
# TODO: store histories in a dict keyed by (optimizer, lr)

In [ ]:
# TODO: plot 1: validation accuracy vs. learning rate (log x-axis), one line per optimizer
# TODO: plot 2: training loss curve at the best learning rate for each optimizer
# TODO: summary table (best val acc per optimizer, best lr)

#### 📝 Discussion of optimizer sensitivity

> TODO: write your interpretation here (a few sentences; this carries marks).
>
- *Which optimizer is most / least sensitive to learning rate? Typical good lr range for each.*
- *Speed of convergence vs. final accuracy; limitations (3 lrs only, few epochs, single seed).*

---
## Task 3: Regularization and residual connections

### 3(a) Regularization study on 5k training images

Small CNN with batch norm; train on a **5k subset** of the training set (validate on the 5k validation split).

Configurations: baseline · weight decay · dropout · flip+crop · cutout · mixup · all combined.

In [ ]:
class SmallBNCNN(nn.Module):
    """TODO: small CNN with BatchNorm; optional dropout probability argument."""
    def __init__(self, p_drop=0.0):
        super().__init__()
        pass
    def forward(self, x):
        pass

**Custom augmentations (must be implemented by you):**

In [ ]:
def random_flip_crop(x, pad=4):
    """TODO: random horizontal flip + random crop with padding (batch-wise, on GPU tensors)."""
    pass

def cutout(x, size=8):
    """TODO: zero out a random size x size patch per image."""
    pass

def mixup(x, y, alpha=1.0):
    """TODO: return mixed inputs, (y_a, y_b), lam. Sample lam ~ Beta(alpha, alpha)."""
    pass

def mixup_criterion(criterion, logits, y_a, y_b, lam):
    pass

In [ ]:
# TODO: visualize each augmentation on a few images (sanity check)

In [ ]:
# TODO: build 5k training subset (seeded, fixed across all runs)
# TODO: run all 7 configurations with identical seed / epochs / lr schedule
#       baseline | weight decay | dropout | flip+crop | cutout | mixup | all combined
# TODO: record histories (train acc, val acc, losses)

In [ ]:
# TODO: table: config | train acc | val acc | gap (train - val)   (use final-epoch or best-epoch consistently; say which)
# TODO: plot train/val curves for all configurations

#### 📝 Interpretation of regularization results

> TODO: write your interpretation here (a few sentences; this carries marks).
>
- *Which methods reduce the generalization gap most? Which improve val accuracy?*
- *Why can train accuracy of mixup / cutout look lower? Do the techniques combine additively?*
- *Caveats: 5k images, single seed, hyperparameters not tuned per method.*

### 3(b) Plain vs. residual CIFAR-style networks, depth $6n+2$ ∈ {8, 20, 56}

Three stages (16/32/64 channels), global average pooling, batch norm, Kaiming init. **The only difference between the two families is the shortcut.**

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride, residual: bool):
        super().__init__()
        # TODO: conv-bn-relu-conv-bn (+ shortcut if residual); decide & document the shortcut type
        #       for dimension changes (1x1 conv vs. padding) - identical for the whole residual family
        pass
    def forward(self, x):
        pass

class CifarNet(nn.Module):
    def __init__(self, n, residual: bool, num_classes=10):
        super().__init__()
        # TODO: depth = 6n+2; stem, 3 stages x n blocks, GAP, FC; Kaiming init
        pass
    def forward(self, x):
        pass

In [ ]:
# TODO: verify depth (count conv/linear layers) and report parameter counts for the 6 models (8/20/56 x plain/residual)

In [ ]:
# TODO: train all 6 models with identical setup (optimizer, lr schedule, epochs, seed)
#       NOTE: total runtime - decide epochs accordingly; record training loss per epoch

In [ ]:
# TODO: plot training loss vs. epoch: plain vs. residual for each depth (e.g. 3 subplots or one figure)
# TODO: summary table: depth | family | final train loss | val acc

#### 📝 Explanation: degradation problem and the role of shortcuts

> TODO: write your interpretation here (a few sentences; this carries marks).
>
- *Does the plain net get worse (higher training loss) as depth grows? Is this overfitting?*
- *How do identity shortcuts ease optimization (gradient flow, identity mapping)?*
- *Role of batch norm + Kaiming init in making plain-56 trainable at all.*

---
## Task 4: CNN (Optional)

Constraints: **≤ 500k parameters**, **≤ 20 training epochs**. Tune on the validation split; evaluate on the test set **exactly once**, at the very end.

### 4.1 Design

> TODO: describe your architecture and design rationale (block type, widths, shortcuts, pooling, ...).
> Include a parameter count and a short justification of each choice.

In [ ]:
class BestNet(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO
        pass
    def forward(self, x):
        pass

# TODO: assert count_parameters(BestNet()) <= 500_000

### 4.2 Training recipe

In [ ]:
# TODO: optimizer, lr schedule (e.g. one-cycle / cosine), weight decay, augmentation, label smoothing, epochs <= 20

### 4.3 Ablation (≥ 3 design choices)

In [ ]:
# TODO: ablation runs, remove / change one design choice at a time vs. the full model
#   e.g. shortcut | augmentation | weight decay | optimizer | ...
# Table: variant | #params | val acc

#### 📝 Discussion of ablation

> TODO: write your interpretation here (a few sentences; this carries marks).
>
- *Which design choice contributes most? Surprises?*
- *Selection done only on validation data - state this explicitly.*

### 4.4 Final evaluation (test set, once)

In [ ]:
# TODO: train/select final model using validation only, then evaluate on the test set ONE time.
# DO NOT re-run this cell after looking at the result or modify the model afterwards.

#### 📝 Final result

> TODO: write your interpretation here (a few sentences; this carries marks).
>
- *Test accuracy vs. validation accuracy; gap, comments on generalization.*

---
## Conclusion

> TODO: summarize the key findings from each task (2–4 bullets per task) and any lessons learned.

## References

> TODO: papers / docs / tools used (e.g. He et al. 2016 ResNet, Kingma & Ba 2015 Adam, Zhang et al. 2018 mixup, DeVries & Taylor 2017 cutout), plus the Claude/Copilot acknowledgement above.